# Tier 2 — the Human branch: which human cause to prevent?

This is one of the three Tier-2 branch deep-dives that hang off the Tier-1 coarse allocator
([`06_analysis.ipynb`](06_analysis.ipynb)). Tier 1 splits a region-season's burned area across
**Human / Natural / Unknown**; this notebook conditions on the **Human** slice and asks the
question that drives *prevention* targeting: **within the human-caused burn, which sub-cause
drives the most acres** — arson, equipment/vehicle use, debris burning, powerlines, recreation,
…? Each sub-cause implies a different intervention, so the deliverable is a **Human sub-cause
risk profile**: for a region and upcoming season, the expected composition of human-caused burn
across the 11 resolved sub-causes, ranked by acres.

**This is the RQ2 forecast partner.** Together with Tier 1, this branch carries the next-season
cause-risk profile; the Natural (location) and Unknown (data-quality) branches are methodologically
distinct sub-projects.

**Method, mirroring Tier 1.** Same grain (EPA Level III ecoregion × meteorological season ×
season-year), same partial-winter boundary rule, same forward-chaining split (score season-year
≥ 2010, train strictly earlier), and the same persistence-floor-first discipline. The target is a
composition on the **11-sub-cause simplex**, conditioned on Human, so it is scored with the same
**acre-weighted total variation distance (TVD)** used for the Tier-1 shares — plus a **top-cause
hit-rate** (does the profile name the right #1 human cause?) for planner legibility.

**Conventions.** Cells run top-to-bottom and are left **unexecuted** for manual run, per project
practice.

In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from config import ProjectConfig
from panel import RegionSeasonPanel

# Project-wide constants (paths, the boundary rule, the forward-chaining split) come
# from one place so this branch cannot drift from the others. See src/config.py.
cfg = ProjectConfig()
DATA = cfg.data

# RegionSeasonPanel applies the same partial-winter boundary rule as 06_analysis. The
# Human branch = all resolved causes except Natural ('Other causes' is a resolved
# determination, just miscellaneous, so it belongs here per the design mapping). The
# missing-cause mass (Unknown) is NOT a cause row and is excluded entirely -- this
# branch conditions on Human.
panel = RegionSeasonPanel.load(cfg)

human = panel.rsc[panel.rsc["cause"] != "Natural"]
SUBCAUSES = sorted(human["cause"].unique())
print(f"{len(human):,} rows | {len(SUBCAUSES)} human sub-causes")
for c in SUBCAUSES:
    print("   -", c)

## Build the Human-conditioned target

One composition per region-season, over the 11 human sub-causes, on a **Human-only denominator**
(total human acres in the cell), so the 11 shares sum to 1. This is deliberately *not* the existing
`cause_share` column — that one is computed over *all attributed* fires (Natural included), so its
denominator carries the lightning mass. Conditioning on Human means re-normalizing within the human
sub-causes alone. Cells with zero human burn carry no composition and are dropped.

In [ ]:
KEYS = list(cfg.cell_keys)

# Shares within Human (sum to 1 across the 11 sub-causes), on a HUMAN-only denominator.
# Deliberately NOT the existing `cause_share` column -- that one is computed over all
# attributed fires (Natural included), so its denominator carries the lightning mass.
# Conditioning on Human means re-normalizing within the human sub-causes alone. Cells
# with zero human burn carry no composition and are dropped.
hc, SHCOLS = panel.human_subcause_shares()

print(f"{len(hc):,} region-season cells with human burn | season_years "
      f"{hc.season_year.min()}-{hc.season_year.max()}")

# Overall human-branch mix (acre-weighted), for orientation.
overall = (hc[SHCOLS].mul(hc["human_total_ac"], axis=0).sum() / hc["human_total_ac"].sum())
print("\nHuman-branch acre-weighted mix (dominant sub-causes first):")
print((overall.sort_values(ascending=False) * 100).round(1).to_string())

## The persistence floor

Same discipline as Tier 1: before any learned model, establish what a trailing mean of the cell's
own past achieves. For each cell, predict the human sub-cause composition as the mean of the last
**k = 7** strictly-prior same-region/same-season occurrences (the window locked for the Tier-1
shares; re-sweeping it for this branch is a later-rung refinement, not a first-pass need).
Forward-chaining throughout — `shift(1)` then a trailing mean, so the target year never enters its
own prediction. Scored on the held-out tail (season-year ≥ 2010) with **acre-weighted TVD** over
the 11-sub-cause simplex, plus a **top-1 hit-rate** on the dominant human cause.

In [ ]:
K = cfg.shares_k                # trailing window locked by the Tier-1 shares sweep
TEST_START = cfg.test_start     # forward-chaining split, shared across all branches

# The forward-chaining rule lives in src/trailing.py rather than being re-typed per
# notebook: TrailingMean does shift(1) then a k-window mean within (region, season) and
# asserts the frame is sorted first. Multi-column, so the whole 11-sub-cause simplex
# is predicted in one call.
from trailing import GlobalPrior, TrailingMean

pred_floor = TrailingMean(K).predict(hc, SHCOLS)

# Predictions are simplex points by construction (mean of simplex points); confirm where defined.
defined = pred_floor.notna().all(axis=1)
assert (pred_floor[defined].sum(axis=1).sub(1).abs() < 1e-9).all(), "floor preds must sum to 1"

actual = hc[SHCOLS].to_numpy()
w = hc["human_total_ac"].to_numpy()                     # acre weights = human acres in the cell
in_test = (hc["season_year"] >= TEST_START).to_numpy()

def tvd_and_hit(P):
    """Acre-weighted TVD over the sub-cause simplex + top-1 dominant-cause hit-rate, on test cells."""
    tvd = 0.5 * np.abs(P - actual).sum(axis=1)
    m = in_test & ~np.isnan(tvd)
    top_hit = (np.nanargmax(P[m], axis=1) == actual[m].argmax(axis=1))
    return {
        "n_cells": int(m.sum()),
        "TVD_unweighted": float(tvd[m].mean()),
        "TVD_acre_wtd": float(np.average(tvd[m], weights=w[m])),
        "top1_hit_unwtd": float(top_hit.mean()),
        "top1_hit_acre_wtd": float(np.average(top_hit, weights=w[m])),
    }

floor = tvd_and_hit(pred_floor.to_numpy())
print(f"HUMAN sub-cause persistence floor (k={K}), held-out tail season_year >= {TEST_START}\n")
for kk, vv in floor.items():
    print(f"  {kk:20s} {vv:.4f}" if isinstance(vv, float) else f"  {kk:20s} {vv:,}")
print("\nTVD: share of the human mix in the wrong sub-cause. top1: did we name the right #1 cause?")

**What the floor achieves.** On the 3,850 held-out human cells, the k=7 trailing mean lands an
acre-weighted TVD of **0.489** — roughly *half* the human mix is placed in the wrong sub-cause — and
names the right #1 human cause in **54%** of cells (acre-weighted). Unweighted the numbers are close
(TVD 0.442, top-1 0.517), so the error is not concentrated in the small cells; the big-burn cells are
about as hard as the rest. This is the number every later rung has to beat.

## Sweeping the window — is k=7 the right trailing length?

The floor above locks **k=7**, inherited from the Tier-1 shares rather than tuned for this Human
branch. The AZ/NM-2011 example exposed the cost of a long window: a single dominant-cause year gets
*diluted* by calmer prior years, so the composition is smeared and the magnitude under-called. A
**shorter** window reacts faster (less smear, but noisier in thin cells); a **longer** one is
steadier (more smear). This rung asks the empirical question directly: across k, does any window beat
k=7 on the same held-out tail, on the same acre-weighted TVD and top-1 hit-rate? The k=7 recipe is
factored into a function of k and swept — everything else (split, weights, scored cells, metrics) is
identical to the floor cell, so the only thing changing is the window length.

In [ ]:
# The floor recipe is now a parameter, not a re-typed block: TrailingMean(k) is the same
# forward-chaining shift(1)-then-trailing-mean used for the k=7 floor above, so this sweep
# changes ONLY the window length. Everything else (split, weights, scored cells, metrics)
# is identical to the floor cell.
K_GRID = [1, 2, 3, 4, 5, 7, 10, 15, 20]
sweep_rows = []
for k in K_GRID:
    P = TrailingMean(k).predict(hc, SHCOLS).to_numpy()
    sweep_rows.append({"k": k, **tvd_and_hit(P)})
sweep = pd.DataFrame(sweep_rows).set_index("k")

best_tvd = sweep["TVD_acre_wtd"].idxmin()
best_hit = sweep["top1_hit_acre_wtd"].idxmax()
print("HUMAN sub-cause persistence — window sweep (held-out tail season_year >= "
      f"{TEST_START}, same {int(sweep['n_cells'].iloc[0]):,} cells)\n")
print(sweep[["TVD_acre_wtd", "TVD_unweighted", "top1_hit_acre_wtd", "top1_hit_unwtd"]].round(4).to_string())
print(f"\nbest acre-wtd TVD at k={best_tvd} ({sweep.loc[best_tvd,'TVD_acre_wtd']:.4f}); "
      f"best top-1 at k={best_hit} ({sweep.loc[best_hit,'top1_hit_acre_wtd']:.4f}). "
      f"locked floor k={K} TVD={sweep.loc[K,'TVD_acre_wtd']:.4f}, top-1={sweep.loc[K,'top1_hit_acre_wtd']:.4f}.")

**Reading the sweep: k=7 sits in a flat basin, and no window buys a real gain.** The short windows
are clearly *worse*, not better — k=1 and k=2 (last one or two years only) score TVD 0.546/0.512 and
top-1 37%/41%. The dilution the AZ/NM-2011 example exposed is real, but reacting to a single prior
year overcorrects into pure noise: one atypical year is a bad forecast. From **k=3 through k=10** the
curve is essentially flat — TVD within ~0.01 and top-1 within ~1 point of each other — then long
windows (k=15, 20) drift worse as over-smoothing sets in.

Within that basin the two metrics point at *different* windows and both "wins" are within noise:
best acre-weighted **TVD is k=3 (0.483)**, a 0.006 improvement over k=7's 0.489 — but k=3's top-1
(53.6%) is a hair *below* k=7's (54.1%); best **top-1 is k=10 (0.544)**, 0.3 points over k=7, but with
slightly worse TVD. Nothing beats k=7 on *both*, and the largest available gain wouldn't change which
cause a planner targets.

**Conclusion: keep k=7.** It's within noise of the best on either metric and sits in the flat part of
the curve, so the Tier-1-inherited window is vindicated here rather than merely convenient. More
importantly, the sweep confirms from a new angle what the learned rungs already showed: the ceiling on
this branch is set by the *information in FPA-FOD*, not by the smoothing window — tuning k doesn't
break through it. The AZ/NM-2011 smearing isn't fixable by window length either, because any window
short enough to sharpen that one big cell wrecks the many thin cells, so the aggregate never improves.
Real lift still has to come from target-closer features or upcoming-season external conditions, not
from this knob.

## A reference point: the global mix

How much does knowing *the region-season* buy over just predicting the same overall human mix
everywhere? This "global prior" baseline predicts every cell as the acre-weighted average human
composition computed from the **training years only** (no leakage). If the persistence floor barely
beats it, human cause is not very regionally structured; if it beats it clearly, region-season
identity carries real prevention signal — the premise of the whole branch.

In [ ]:
train = (hc["season_year"] < TEST_START).to_numpy()

# One acre-weighted composition for every cell, fit on training years only -- the
# region-season-blind reference. Same GlobalPrior used by the Natural and Unknown branches.
P_global = (GlobalPrior(weighted=True)
            .fit(hc, SHCOLS, train_mask=train, weight_col="human_total_ac")
            .predict(hc).to_numpy())
glob = tvd_and_hit(P_global)

cmp = pd.DataFrame([glob, floor], index=["global prior (train mix)", f"persistence (k={K})"])
cmp["TVD_delta_vs_global"] = cmp["TVD_acre_wtd"] - glob["TVD_acre_wtd"]
print(cmp[["n_cells", "TVD_acre_wtd", "top1_hit_acre_wtd", "TVD_delta_vs_global"]].round(4).to_string())
print("\nnegative delta = persistence (region-season aware) beats the one-size-fits-all mix.")

**Region-season identity carries the signal.** The one-size-fits-all global mix scores TVD **0.643**
and names the right top cause only **16%** of the time; the region-season-aware persistence floor cuts
TVD by **0.154** (to 0.489) and more than triples the top-1 hit-rate (to 54%). So *where and when* a
fire burns tells you a great deal about *which* human cause dominates — the premise of this whole
branch holds. The open question the next rung tests is whether anything beyond the cell's own past adds
to that.

## A concrete profile

The deliverable as a planner reads it: for the largest-burn held-out human cell, the predicted
human sub-cause profile ranked by acres, against what actually burned. This is the object the
prevention recommendation is built on — the top row is the cause to target.

In [14]:
Pf = pred_floor.to_numpy()
scored = in_test & pred_floor.notna().all(axis=1).to_numpy()
idx = np.where(scored)[0]
big = idx[np.argmax(w[idx])]                            # largest human-burn held-out cell
row = hc.iloc[big]

prof = pd.DataFrame({
    "predicted_share": Pf[big],
    "actual_share": actual[big],
    "predicted_acres": Pf[big] * row["human_total_ac"],
    "actual_acres": actual[big] * row["human_total_ac"],
}, index=[c.replace("sh_", "") for c in SHCOLS]).sort_values("predicted_acres", ascending=False)

print(f"Human sub-cause profile -- {row['region']} / {row['season']} / season-year {int(row['season_year'])}")
print(f"(total human burn in cell: {row['human_total_ac']:,.0f} ac)\n")
print(prof.round(3).to_string())

Human sub-cause profile -- Arizona/New Mexico Mountains / MAM / season-year 2011
(total human burn in cell: 729,284 ac)

                                            predicted_share  actual_share  predicted_acres  actual_acres
Recreation and ceremony                               0.447         0.934       325643.233     680921.26
Equipment and vehicle use                             0.266         0.001       194208.022       1000.75
Debris and open burning                               0.159         0.001       116153.445        474.43
Power generation/transmission/distribution            0.076         0.044        55197.027      31893.47
Misuse of fire by a minor                             0.029         0.000        21199.284         54.00
Arson/incendiarism                                    0.019         0.006        14211.089       4318.42
Smoking                                               0.003         0.000         2281.773        220.70
Railroad operations and maintenance    

**What this example shows — and where it's hard.** The largest held-out human cell is the Arizona/New
Mexico Mountains, spring (MAM) 2011: ~729k acres of human burn. The floor's top pick is **Recreation
and ceremony** (predicted 45%), and that *is* the actual #1 — it took 93% of the real burn. So the
top-1 call is right and the prevention headline (target recreation ignitions) is correct. But the
*magnitude* is badly under-called (45% predicted vs 93% actual), and the floor spreads the remainder
across Equipment/Debris that barely burned, while missing a real 1.4% Fireworks tail. The lesson the
metrics already flag: even on a cell where persistence names the right cause, the composition is smeared
— a single dominant real cause gets diluted by the trailing average of quieter prior years. This is
exactly the kind of error a sharper model would have to fix.

## The learned cross-sectional rung — does "what kind of region-season is this?" beat persistence?

The floor above predicts a cell's human sub-cause mix purely from **its own past** (the k=7 trailing
mean). The next rung, exactly as in Tier 1 ([`06_analysis.ipynb`](06_analysis.ipynb)), looks for lift
**cross-sectionally** instead: predict a region-season's human composition from *what kind of
region-season it is*, letting similar cells inform each other, using the trailing "fingerprint"
features from [`05_features.ipynb`](05_features.ipynb) (`data/region_season_features.parquet`).

Every fingerprint feature there is a strictly-prior trailing summary (forward-chaining, leakage-audited
in that notebook), plus `season` as a non-leaking identity feature. Note the fingerprints are *coarse*
(Tier-1) trailing summaries — they describe the region-season's character (how much it burns, its
long-run Human/Natural/Unknown split), **not** its human sub-cause history — so this rung tests whether
that character predicts the **within-Human** mix at all. We fit one gradient-boosted regressor per
sub-cause on the **same forward-chaining split** (train `season_year < 2010`, score `>= 2010`),
renormalize onto the 11-sub-cause simplex, and grade with the **exact same** acre-weighted TVD and
top-1 hit-rate as the floor — a head-to-head on identical held-out cells. The interpretation follows
the run.

In [15]:
# Load the cross-sectional fingerprint table (built + leakage-audited in 05_features.ipynb).
feats = pd.read_parquet(DATA / "region_season_features.parquet")

KEY = ["region", "season", "season_idx"]

# Attach the Human target (11 sub-cause shares + human acres) and the locked k=7 floor prediction
# onto the feature rows, keyed (not positional), so the head-to-head is guaranteed same-cell. Inner
# join: only cells that have both a human composition and a fingerprint are comparable.
floor_df = hc[KEY].copy()
for j, c in enumerate(SHCOLS):
    floor_df[f"floor_{c}"] = pred_floor.to_numpy()[:, j]   # k=7 trailing-mean human shares
tgt = hc[KEY + SHCOLS + ["human_total_ac", "season_year"]].merge(floor_df, on=KEY, how="inner")
feats = tgt.merge(feats.drop(columns=["season_year"], errors="ignore"), on=KEY, how="inner")

FEATCOLS = [c for c in feats.columns if c.startswith("f_")]

# season is a legitimate non-leaking identity feature (a cell knows its own season); one-hot it.
X_all = pd.get_dummies(feats[FEATCOLS + ["season"]], columns=["season"], dtype=float)
XCOLS = list(X_all.columns)

# Same forward-chaining split as the floor: train strictly before 2010, score 2010+. A cell is
# usable only where its trailing fingerprint AND its floor prediction exist (first-occurrence cells
# are NaN on both -- so both sides of the comparison see the identical held-out tail).
has_feat  = feats[FEATCOLS].notna().all(axis=1).to_numpy()
has_floor = feats[[f"floor_{c}" for c in SHCOLS]].notna().all(axis=1).to_numpy()
usable = has_feat & has_floor
tr = (feats["season_year"] < TEST_START).to_numpy() & usable
te = (feats["season_year"] >= TEST_START).to_numpy() & usable

wtf = feats["human_total_ac"].to_numpy()               # acre weights = human acres (same as the floor)
print(f"features: {len(XCOLS)} cols ({len(FEATCOLS)} fingerprint + season one-hot)")
print(f"train cells (<{TEST_START}, with history): {tr.sum():,}")
print(f"test  cells (>={TEST_START}, with history): {te.sum():,}")

features: 12 cols (8 fingerprint + season one-hot)
train cells (<2010, with history): 5,312
test  cells (>=2010, with history): 3,846


In [16]:
# LEARNED HUMAN SHARES: one gradient-boosted regressor per sub-cause, predictions clipped and
# renormalized onto the 11-sub-cause simplex so they sum to 1 like the target. Same acre-weighted
# TVD + top-1 hit-rate as the floor, on the SAME held-out cells.
from sklearn.ensemble import HistGradientBoostingRegressor

Xtr, Xte = X_all.to_numpy()[tr], X_all.to_numpy()[te]
Ytr = feats[SHCOLS].to_numpy()[tr]

pred_parts = []
for j in range(len(SHCOLS)):
    m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                      max_leaf_nodes=31, random_state=0)
    m.fit(Xtr, Ytr[:, j], sample_weight=wtf[tr])       # weight training by human acres too
    pred_parts.append(m.predict(Xte))
P_learn = np.clip(np.vstack(pred_parts).T, 0, None)
P_learn = P_learn / P_learn.sum(axis=1, keepdims=True)  # renormalize onto the simplex

A_te  = feats[SHCOLS].to_numpy()[te]                    # actual human shares on the test cells
w_te  = wtf[te]
P_flr = feats[[f"floor_{c}" for c in SHCOLS]].to_numpy()[te]   # key-aligned k=7 floor, same cells

def score(P):
    tvd = 0.5 * np.abs(P - A_te).sum(axis=1)
    hit = (P.argmax(axis=1) == A_te.argmax(axis=1))
    return {
        "TVD_unweighted":    tvd.mean(),
        "TVD_acre_wtd":      np.average(tvd, weights=w_te),
        "top1_hit_acre_wtd": np.average(hit, weights=w_te),
    }

rung = pd.DataFrame([score(P_flr), score(P_learn)],
                    index=[f"persistence floor (k={K})", "learned (gradient boosting)"])
rung["TVD_delta_vs_floor"] = rung["TVD_acre_wtd"] - score(P_flr)["TVD_acre_wtd"]
print(f"HUMAN sub-cause — head-to-head on {te.sum():,} held-out cells (season_year >= {TEST_START})\n")
print(rung.round(4).to_string())
print("\nlower TVD = better. negative delta = the fingerprint beats the trailing mean.")

HUMAN sub-cause — head-to-head on 3,846 held-out cells (season_year >= 2010)

                             TVD_unweighted  TVD_acre_wtd  top1_hit_acre_wtd  TVD_delta_vs_floor
persistence floor (k=7)              0.4417        0.4887             0.5405               0.000
learned (gradient boosting)          0.5413        0.5877             0.3566               0.099

lower TVD = better. negative delta = the fingerprint beats the trailing mean.


## Reading the head-to-head: the coarse fingerprint loses

The learned rung does not just fail to add signal — it is **worse** than the cell's own trailing mean on
both metrics. Acre-weighted TVD rises from the floor's **0.489** to **0.588** (a delta of **+0.099**),
and the top-1 dominant-cause hit-rate *falls* from **54%** to **36%**. The coarse fingerprint is
actively misleading here, not merely uninformative.

The reading: a region's **coarse character** — how much it burns, its long-run Human/Natural/Unknown
split — does **not** recover its **within-Human** sub-cause mix. That mix is better carried by the
cell's own human history, which the floor uses and this feature set deliberately does not. Cross-cell
pooling on the wrong features costs accuracy rather than borrowing it.

The remaining lift, if any, lives in one of two places, neither in this feature set:

- **trailing *human-sub-cause* fingerprints** — a feature set built from the cell's own past human mix,
  much closer to the target than the coarse Tier-1 summaries used here; or
- **upcoming-season conditions** (weather, drought, local land use) not present in FPA-FOD at all — the
  same external-data seam the Natural (location) and Unknown (data-quality) branches open.

For now the honest result stands: **on internal FPA-FOD data, the k=7 persistence floor is the model to
beat, and the coarse cross-sectional rung does not beat it.**

## The next rung — give the model the cell's own human history

The coarse rung lost because it never saw the cell's *human sub-cause* past — only its Tier-1 character.
This rung fixes exactly that: we hand the gradient booster the **k=7 trailing human mix itself** (the 11
`floor_*` columns — the very quantity the persistence floor predicts) as **features**, alongside the 8
coarse fingerprints and `season`. Those columns are already strictly-prior (`shift(1)` then trailing
mean, built in the floor cell), so they carry no leakage.

This is the clean scientific test recommended over a blend: if a model that can *combine* own-history
with cross-cell pooling still can't beat own-history-used-raw (the floor), then on internal FPA-FOD data
the floor is the ceiling and any further lift must come from external conditions. Same forward-chaining
split, same held-out cells, same acre-weighted TVD + top-1 — a three-way head-to-head: floor vs. the
coarse learned rung vs. this history-aware learned rung. Interpretation follows the run.

In [17]:
# HISTORY-AWARE LEARNED RUNG: same GBR-per-sub-cause recipe as above, but the feature set now also
# includes the 11 k=7 trailing human-mix columns (floor_*) -- the cell's own human sub-cause history.
# Everything else (split, weights, held-out cells, scoring) is identical, so the only thing that
# changes vs. the coarse rung is *what the model gets to see*.
HIST_COLS = [f"floor_{c}" for c in SHCOLS]              # 11 strictly-prior trailing human shares

X_hist = pd.concat([X_all, feats[HIST_COLS]], axis=1)   # coarse fingerprints + season + human history
Xtr_h, Xte_h = X_hist.to_numpy()[tr], X_hist.to_numpy()[te]

pred_parts_h = []
for j in range(len(SHCOLS)):
    m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                      max_leaf_nodes=31, random_state=0)
    m.fit(Xtr_h, Ytr[:, j], sample_weight=wtf[tr])
    pred_parts_h.append(m.predict(Xte_h))
P_hist = np.clip(np.vstack(pred_parts_h).T, 0, None)
P_hist = P_hist / P_hist.sum(axis=1, keepdims=True)     # renormalize onto the 11-sub-cause simplex

# Three-way head-to-head on the identical held-out cells.
three = pd.DataFrame(
    [score(P_flr), score(P_learn), score(P_hist)],
    index=[f"persistence floor (k={K})", "learned (coarse fingerprints)", "learned (+ human history)"],
)
three["TVD_delta_vs_floor"] = three["TVD_acre_wtd"] - score(P_flr)["TVD_acre_wtd"]
print(f"HUMAN sub-cause — three-way on {te.sum():,} held-out cells (season_year >= {TEST_START})")
print(f"history-aware feature set: {X_hist.shape[1]} cols "
      f"({len(FEATCOLS)} coarse + season one-hot + {len(HIST_COLS)} human-history)\n")
print(three.round(4).to_string())
print("\nlower TVD = better. negative delta vs floor = the learned rung finally beats the trailing mean.")

HUMAN sub-cause — three-way on 3,846 held-out cells (season_year >= 2010)
history-aware feature set: 23 cols (8 coarse + season one-hot + 11 human-history)

                               TVD_unweighted  TVD_acre_wtd  top1_hit_acre_wtd  TVD_delta_vs_floor
persistence floor (k=7)                0.4417        0.4887             0.5405              0.0000
learned (coarse fingerprints)          0.5413        0.5877             0.3566              0.0990
learned (+ human history)              0.5110        0.5536             0.4748              0.0649

lower TVD = better. negative delta vs floor = the learned rung finally beats the trailing mean.
